In [ ]:
import pandas as pd
import torch
import numpy as np
import re
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    pipeline
)
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

In [ ]:
df = pd.read_excel(r'Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\salud_completo.xlsx')

# Eliminar ids duplicados si los hay

df_texto = df[
    df['ES - Corredores - Verbatim Comment including Corredores']
    .notna() & 
    (df['ES - Corredores - Verbatim Comment including Corredores'].str.strip() != "")
]

print("Filas totales:", len(df))
print("Filas con texto:", len(df))

Analisis de sentimiento

In [ ]:
# ======================================================
# 1. CONFIGURACIÓN GENERAL
# ======================================================
RUTA_MODELO = r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\beto-sentiment"
col_texto = "ES - Corredores - Verbatim Comment including Corredores"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================================
# 2. CARGA MODELOS
# ======================================================
tokenizer = AutoTokenizer.from_pretrained(RUTA_MODELO, local_files_only=True)

model_cls = AutoModelForSequenceClassification.from_pretrained(
    RUTA_MODELO, local_files_only=True
).to(device)
model_cls.eval()

model_emb = AutoModel.from_pretrained(
    RUTA_MODELO, local_files_only=True
)
model_emb.eval()

# ======================================================
# 3. PREPROCESADO TEXTO
# ======================================================
df_texto = df_texto.copy()

def normalizar_texto(texto):
    texto = str(texto).strip()
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.lower()
    return texto

df_texto["texto"] = df_texto[col_texto].astype(str)
df_texto["texto_norm"] = df_texto["texto"].apply(normalizar_texto)

# ======================================================
# 4. CLASIFICACIÓN BASE (BETO)
# ======================================================
def tokenize_batch(textos):
    return tokenizer(
        textos,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

batch_size = 32
loader = DataLoader(
    df_texto["texto_norm"].tolist(),
    batch_size=batch_size
)

sentimientos = []
confianzas = []

with torch.no_grad():
    for batch in loader:
        inputs = tokenize_batch(batch).to(device)
        outputs = model_cls(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1)
        conf = probs.max(dim=1).values

        sentimientos.extend(pred.tolist())
        confianzas.extend(conf.tolist())

mapa = {
    0: "Negativo",
    1: "Positivo",
    2: "Neutro"
}

df_texto["sentimiento_base_num"] = sentimientos
df_texto["sentimiento_base"] = df_texto["sentimiento_base_num"].map(mapa)
df_texto["confianza"] = confianzas

# ======================================================
# 5. EMBEDDINGS Y PROTOTIPOS
# ======================================================
def obtener_embedding(texto):
    texto = normalizar_texto(texto)
    inputs = tokenizer(
        texto,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model_emb(**inputs)

    return outputs.last_hidden_state[:, 0, :].numpy()[0]

def normalizar(v):
    return v / np.linalg.norm(v)

proto_positivo = [
    "la atención fue adecuada",
    "el servicio ha sido satisfactorio",
    "todo se resolvió correctamente",
    "buena atención recibida",
    "gestión correcta del caso"
]

proto_neutro = [
    "se realizó la gestión",
    "información facilitada",
    "consulta registrada",
    "sin incidencias destacables",
    "proceso estándar"
]

proto_negativo = [
    "problemas con la atención",
    "no se resolvió el problema",
    "incidencias en el servicio",
    "experiencia insatisfactoria",
    "mala gestión del caso"
]

def embedding_medio(frases):
    embs = [normalizar(obtener_embedding(f)) for f in frases]
    return np.mean(embs, axis=0)

emb_pos = embedding_medio(proto_positivo)
emb_neu = embedding_medio(proto_neutro)
emb_neg = embedding_medio(proto_negativo)

# ======================================================
# 6. RECLASIFICACIÓN DE NEUTROS
# ======================================================
def reclasificar_neutro(texto, delta=0.08):
    emb = normalizar(obtener_embedding(texto))

    sim_pos = cosine_similarity([emb], [emb_pos])[0][0]
    sim_neu = cosine_similarity([emb], [emb_neu])[0][0]
    sim_neg = cosine_similarity([emb], [emb_neg])[0][0]

    sims = {
        "Positivo_leve": sim_pos,
        "Neutro_real": sim_neu,
        "Negativo_leve": sim_neg
    }

    etiqueta = max(sims, key=sims.get)
    valores = sorted(sims.values(), reverse=True)

    if valores[0] - valores[1] < delta:
        return "Neutro_real"

    return etiqueta

mask_neutros = df_texto["sentimiento_base"] == "Neutro"

df_texto.loc[mask_neutros, "sub_sentimiento"] = (
    df_texto.loc[mask_neutros, "texto"]
    .apply(reclasificar_neutro)
)

# ======================================================
# 7. SENTIMIENTO FINAL
# ======================================================
def sentimiento_final(row):
    base = row["sentimiento_base"]
    sub = row.get("sub_sentimiento", np.nan)

    if base in ["Positivo", "Negativo"]:
        return base

    if sub == "Positivo_leve":
        return "Positivo"
    elif sub == "Negativo_leve":
        return "Negativo"
    else:
        return "Neutro"

df_texto["sentimiento_final"] = df_texto.apply(sentimiento_final, axis=1)

Ahora hacemos un merge entre el dataframe creado y el original para que todas las columnas creadas aparezcan en el original y creamos un archivo excel que se llame: nps_salud_analisis_sentimiento.xlsx

In [ ]:
cols_sentimiento = [
    "Surveyid for internal use (e.g. RI link)",
    "sentimiento_base_num",
    "sentimiento_base",
    "confianza",
    "sentimiento_final"
]

df_texto_merge = df_texto[cols_sentimiento].copy()


df_final = df.merge(
    df_texto_merge,
    on="Surveyid for internal use (e.g. RI link)",
    how="left"
)


ruta_salida = r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\nps_salud_analisis_sentimiento.xlsx"

df_final.to_excel(
    ruta_salida,
    index=False,
    engine="openpyxl"
)

Compronbamos que haya el mismo número de comentarios que de sentimientos

In [ ]:
col_texto = "ES - Corredores - Verbatim Comment including Corredores"

num_comentarios = df[
    df[col_texto].notna() & (df[col_texto].str.strip() != "")
].shape[0]
num_sentimientos = df_final["sentimiento_final"].notna().sum()


print("Filas con comentario:", num_comentarios)
print("Filas con sentimiento final:", num_sentimientos)

if num_comentarios == num_sentimientos:
    print("OK: coinciden exactamente")
else:
    print("ATENCIÓN: no coinciden")

Analisis de aspectos

In [ ]:
import pandas as pd
import torch
from collections import defaultdict
from transformers import pipeline

RUTA_MODELO = r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\beto-sentiment"

device_id = 0 if torch.cuda.is_available() else -1
batch_size = 32

aspectos = {
    'Servicio': [
        'profesionalidad', 'profesional', 'buen profesional', 'gran profesional', 'mal profesional',
        'trato', 'trato humano', 'trato excelente', 'trato amable', 'buen trato', 'mal trato',
        'amabilidad', 'amable', 'desagradable', 'empatía', 'cercanía', 'poca empatía', 'antipáticos',
        'explicaciones', 'explicar', 'explicaciones claras', 'mala explicación', 'no explican',
        'información', 'informar', 'información clara', 'sin información',
        'atención', 'mala atención', 'atención deficiente', 'falta de atención', 'buena atención',
        'urgencias', 'urgente', 'emergencia',
        'hospitalización', 'hospital', 'ingreso', 'postoperatorio', 'cirugía', 'operación',
        'fisioterapia', 'rehabilitación', 'fisioterapeuta',
        'podología', 'podólogo', 'logopeda', 'logopedia',
        'diagnóstico', 'erróneo', 'tardío', 'incorrecto',
        'médico', 'doctor', 'doctor malo', 'buen médico', 'mal médico',
        'consulta', 'consulta médica', 'visita', 'revisión'
    ],
    'Precio': [
        'precio', 'caro', 'carísimo', 'muy caro', 'precio alto', 'precio abusivo', 'precio elevado',
        'cuota', 'cuota cara', 'cuota elevada', 'subida', 'subida de precio', 'subida excesiva',
        'incremento', 'incremento de cuota', 'prima', 'prima elevada',
        'copago', 'copago excesivo', 'copagos',
        'coste', 'coste elevado', 'costo',
        'facturación', 'factura', 'cobro', 'cobro indebido', 'facturación incorrecta',
        'relación calidad precio', 'pagar', 'pago', 'excesivo', 'abusivo'
    ],
    'Producto': [
        'cuadro médico', 'cuadro', 'limitado', 'insuficiente',
        'especialista', 'especialistas', 'falta de especialistas', 'no hay especialista',
        'dermatólogo', 'reumatólogo', 'podólogo', 'oftalmólogo', 'cardiólogo', 'neurólogo',
        'cobertura', 'cobertura insuficiente', 'exclusiones', 'no cubre', 'carencias',
        'autorizado', 'no autorizado', 'autorizar', 'no aprueban',
        'dental', 'dentista', 'diente', 'limpieza dental',
        'pruebas', 'pruebas diagnósticas', 'prueba',
        'máquina', 'antiguo', 'obsoleto',
        'sesiones', 'sesión', 'limitadas'
    ],
    'Velocidad': [
        'espera', 'lista de espera', 'tiempo de espera', 'mucho tiempo', 'mucha espera',
        'meses', 'mes', 'semanas', 'semana', 'días', 'día',
        'demora', 'demoras', 'retraso', 'retrasos', 'retrasado',
        'cita', 'citas', 'cita tardía', 'retraso cita',
        'tarde', 'tardío', 'tardanza', 'lento', 'lenitud',
        'autorización lenta', 'reembolso lento', 'lentitud',
        'agilidad', 'ágil', 'rápido', 'rápida', 'rapidez',
        'sin esperas', 'puntual', 'puntualidad', 'no puntual',
        'pronto', 'urgencia', 'urgente'
    ],
    'Sistema': [
        'app', 'aplicación', 'aplicación móvil',
        'app no funciona', 'app lenta', 'app poco intuitiva', 'app difícil',
        'web', 'página web', 'portal', 'plataforma',
        'web complicada', 'web difícil', 'fallos web', 'fallo',
        'sistema', 'problemas técnicos', 'problema técnico',
        'agenda', 'agendas cerradas', 'cita online', 'no hay citas',
        'gestión burocrática', 'burocracia', 'burocrático',
        'online', 'oficio', 'teleasistencia'
    ],
    'Comunicación': [
        'información', 'informar', 'falta de información', 'no informan', 'sin información',
        'comunicación', 'mala comunicación', 'comunicar',
        'información contradictoria', 'información confusa', 'confuso',
        'respuesta', 'respuestas automáticas', 'sin respuesta', 'no responden',
        'robot', 'automatizado',
        'contacto', 'dificultad para contactar', 'contactar',
        'teléfono', 'llamada', 'llamar', 'no cogen', 'no cogen teléfono',
        'seguimiento', 'sin seguimiento', 'seguir',
        'aviso', 'avisan', 'no avisan', 'notificación',
        'email', 'correo', 'sms', 'mensaje',
        'atendedor', 'recepción'
    ]
}

# pipeline (igual que antes)
aspect_model = pipeline(
    "sentiment-analysis",
    model=RUTA_MODELO,
    tokenizer=RUTA_MODELO,
    local_files_only=True,
    device=device_id,
    batch_size=batch_size
)

def find_aspects(text, aspectos):
    text_lower = text.lower()
    return [category for category, keywords in aspectos.items() if any(keyword in text_lower for keyword in keywords)]

# FILTRAR SOLO FILAS CON TEXTO
col_texto = "ES - Corredores - Verbatim Comment including Corredores"

df = pd.read_excel(r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\nps_salud_analisis_sentimiento.xlsx", dtype=object)

df_filtrado = df[df[col_texto].notna() & (df[col_texto].str.strip() != "")].copy()

df_filtrado["texto"] = df_filtrado[col_texto].astype(str)
df_filtrado["aspect_matches"] = df_filtrado["texto"].apply(lambda t: find_aspects(t, aspectos))

# (opcional) función para recortar texto
def recortar_texto(texto, max_chars=2000):
    return texto[:max_chars]

rows = []
for idx, aspects_list in df_filtrado["aspect_matches"].items():
    text = df_filtrado.at[idx, "texto"]
    for aspect in aspects_list:
        rows.append({
            "idx": idx,
            "aspect": aspect,
            "input_text": f"{aspect}. {recortar_texto(text)}"
        })

predictions = []
rows_texts = [row["input_text"] for row in rows]

for i in range(0, len(rows_texts), batch_size):
    batch = rows_texts[i:i + batch_size]

    predictions.extend(
        aspect_model(
            batch,
            truncation=True,
            max_length=512
        )
    )

results = defaultdict(list)
for row, pred in zip(rows, predictions):
    results[row["idx"]].append(str(row["aspect"]))

df_filtrado["aspectos_absa"] = df_filtrado.index.map(lambda idx: results.get(idx, []))

df_filtrado = df_filtrado.drop(columns=["aspect_matches"], errors="ignore")

# columnas binarias
for aspecto_categoria in aspectos.keys():
    columna = f"aspecto_{aspecto_categoria}"
    df_filtrado[columna] = df_filtrado["aspectos_absa"].apply(
        lambda lista_aspectos: int(
            isinstance(lista_aspectos, list) and aspecto_categoria in lista_aspectos
        )
    )

df_filtrado["aspectos_count"] = df_filtrado["aspectos_absa"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# asegurar columnas en df original
for col in df_filtrado.columns:
    if col not in df.columns:
        df[col] = None

# actualizar solo filas con texto
df.update(df_filtrado)

# guardar resultado
df.to_excel(r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\nps_salud_sentimiento_y_aspeccto.xlsx", index=False)

print("Script ejecutado correctamente")